# BLOX_2: Sequential Greedy バッチ選択による実験候補の多様化

blox_1.ipynb の拡張版。複数の実験候補を選ぶ際に単純な上位N件を選ぶのではなく、
**Sequential Greedy** 法により空間的に多様な候補セットを選択します。

### Sequential Greedy の考え方

候補を1件選択するたびに、その点を「仮の既知点」として追加し Stein Novelty を再計算します。
これにより、同じ高novelty領域から重複して選ばれることを防ぎ、探索空間の多様性を確保します。

```
Step 1: 全未測定点のnoveltyを計算 → 最高novelty点を選択（候補#1）
Step 2: 候補#1を仮の既知点に追加 → noveltyを再計算 → 最高novelty点を選択（候補#2）
Step 3: 候補#2も仮の既知点に追加 → noveltyを再計算 → 最高novelty点を選択（候補#3）
  ...
```

## CSV フォーマット（想定）

| 列名 | 内容 |
|---|---|
| `Temp_1`, `Temp_2`, `Temp_3` | 記述子（温度条件など） |
| `V_cut_1`, `V_cut_2`, `V_cut_3` | 記述子（電圧条件など） |
| `Target` | 目的変数。**未測定の行は空欄（NaN）のままにしておく** |

## 1. ライブラリ読み込み

In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import KFold
from scipy.spatial.distance import cdist
import matplotlib.pyplot as plt
import os

## 2. ユーザー設定 ⚙️

**このセルの値だけを変更してください。それ以外のセルは編集不要です。**

| 変数 | 説明 |
|---|---|
| `CSV_PATH` | 読み込む CSV ファイルのパス |
| `FEATURE_COLS` | 記述子（説明変数）の列名リスト |
| `TARGET_COL` | 目的変数の列名 |
| `N_BATCH` | 次に実験する候補の件数（Sequential Greedy で選択） |
| `N_ESTIMATORS` | Random Forest の木の本数 |
| `RANDOM_STATE` | 乱数シード（再現性確保用） |
| `STEIN_KERNEL_BANDWIDTH` | Stein novelty 計算に使う RBF カーネルのバンド幅。`"median"` で中央値ヒューリスティックを自動使用 |
| `N_CV_FOLDS` | 既知データに対する交差検証の分割数（モデル性能チェック用） |
| `OUTPUT_DIR` | CSV・PNG の保存先ディレクトリ |

In [ ]:
# ===== ユーザー設定 =====
CSV_PATH     = "battery_data.csv"   # 読み込む CSV ファイルのパス

FEATURE_COLS = ['Temp_1', 'Temp_2', 'Temp_3', 'V_cut_1', 'V_cut_2', 'V_cut_3']
TARGET_COL   = 'Target'

N_BATCH      = 5    # 次に実験する候補の件数（Sequential Greedy で選択）

N_ESTIMATORS = 200
RANDOM_STATE = 42

STEIN_KERNEL_BANDWIDTH = "median"   # 数値を直接指定することも可能 (例: 1.0)
N_CV_FOLDS   = 5

OUTPUT_DIR   = "."  # 出力先フォルダ（必要に応じて変更可）
os.makedirs(OUTPUT_DIR, exist_ok=True)

## 3. データ読み込みと分割

In [ ]:
df = pd.read_csv(CSV_PATH)

print(f"読み込んだ全行数: {len(df)}")
print(f"記述子: {FEATURE_COLS}")
print(f"目的変数: {TARGET_COL}")

missing_cols = [c for c in FEATURE_COLS + [TARGET_COL] if c not in df.columns]
if missing_cols:
    raise ValueError(f"CSVに以下の列が見つかりません: {missing_cols}")

df.head()

In [ ]:
is_known = df[TARGET_COL].notna()

df_known   = df[is_known].reset_index(drop=True)
df_unknown = df[~is_known].reset_index(drop=True)

print(f"既知データ（回帰に使用）: {len(df_known)} 件")
print(f"未測定データ（novelty評価対象）: {len(df_unknown)} 件")

if len(df_known) < 2:
    raise ValueError("回帰モデル構築には目的変数が入力された行が2件以上必要です。")
if len(df_unknown) == 0:
    print("注意: 未測定（Target空欄）のデータが見つかりません。全行に値が入力されています。")
if N_BATCH > len(df_unknown):
    print(f"警告: N_BATCH ({N_BATCH}) が未測定データ件数 ({len(df_unknown)}) を超えています。N_BATCH を調整します。")
    N_BATCH = len(df_unknown)

X_known   = df_known[FEATURE_COLS].values
y_known   = df_known[TARGET_COL].values
X_unknown = df_unknown[FEATURE_COLS].values

## 4. 回帰モデルの構築（既知データのみ）

Random Forest を全既知データで学習します。あわせて交差検証で予測性能の目安（RMSE, R²）を確認します。

In [ ]:
def evaluate_cv(X, y, n_splits, n_estimators, random_state):
    n_splits_eff = min(n_splits, len(X))
    if n_splits_eff < 2:
        print("交差検証をスキップ（データ点数が不足しています）")
        return None, None

    kf = KFold(n_splits=n_splits_eff, shuffle=True, random_state=random_state)
    rmses, r2s = [], []

    for train_idx, test_idx in kf.split(X):
        model = RandomForestRegressor(n_estimators=n_estimators, random_state=random_state, n_jobs=-1)
        model.fit(X[train_idx], y[train_idx])
        preds = model.predict(X[test_idx])
        rmses.append(np.sqrt(mean_squared_error(y[test_idx], preds)))
        if len(test_idx) > 1:
            r2s.append(r2_score(y[test_idx], preds))

    return np.array(rmses), (np.array(r2s) if r2s else None)

cv_rmse, cv_r2 = evaluate_cv(X_known, y_known, N_CV_FOLDS, N_ESTIMATORS, RANDOM_STATE)

if cv_rmse is not None:
    print(f"CV RMSE: {cv_rmse.mean():.4f} +/- {cv_rmse.std():.4f}")
    if cv_r2 is not None:
        print(f"CV R²  : {cv_r2.mean():.4f} +/- {cv_r2.std():.4f}")

In [ ]:
final_model = RandomForestRegressor(n_estimators=N_ESTIMATORS, random_state=RANDOM_STATE, n_jobs=-1)
final_model.fit(X_known, y_known)
print("最終モデルの学習が完了しました。")

## 5. Stein Novelty の計算関数

In [ ]:
def compute_bandwidth(X_ref, mode="median"):
    if isinstance(mode, (int, float)):
        return float(mode)
    if mode == "median":
        dists = cdist(X_ref, X_ref, metric='euclidean')
        iu = np.triu_indices_from(dists, k=1)
        med = np.median(dists[iu])
        return med if med > 0 else 1.0
    raise ValueError(f"未対応のbandwidthモード: {mode}")


def rbf_kernel(X1, X2, bandwidth):
    sq_dists = cdist(X1, X2, metric='sqeuclidean')
    return np.exp(-sq_dists / (2.0 * bandwidth ** 2))


def predictive_variance(model, X):
    tree_preds = np.array([tree.predict(X) for tree in model.estimators_])
    return np.var(tree_preds, axis=0), np.mean(tree_preds, axis=0)


def stein_novelty(X_candidates, X_reference, model, bandwidth_mode="median", eps=1e-8):
    """
    novelty(x) = predictive_variance(x) / sum_j k(x, x_j)

    X_reference には実測既知点だけでなく、Sequential Greedy で選択済みの
    仮想既知点も含めて渡す。
    """
    mu    = X_reference.mean(axis=0)
    sigma = X_reference.std(axis=0)
    sigma[sigma == 0] = 1.0

    X_ref_scaled  = (X_reference   - mu) / sigma
    X_cand_scaled = (X_candidates  - mu) / sigma

    bandwidth = compute_bandwidth(X_ref_scaled, bandwidth_mode)

    K       = rbf_kernel(X_cand_scaled, X_ref_scaled, bandwidth)
    density = K.sum(axis=1)

    pred_var, pred_mean = predictive_variance(model, X_candidates)
    novelty = pred_var / (density + eps)

    return novelty, pred_mean, pred_var, density, bandwidth

## 6. Sequential Greedy バッチ選択

候補を1件選ぶたびにその点を**仮の既知点**として参照セットに追加し、
Stein Novelty を再計算してから次の候補を選びます。
これにより高novelty領域が集中していても、選択済みの点の周辺は
カーネル密度が上昇して novelty が下がるため、自動的に多様な候補が選ばれます。

In [ ]:
if len(df_unknown) > 0:
    # 参照セット（実測既知点）と残り候補のインデックス管理
    X_ref_virtual  = X_known.copy()          # 仮の既知セット（随時拡張）
    remaining_mask = np.ones(len(X_unknown), dtype=bool)  # まだ選ばれていない候補

    selected_indices   = []   # X_unknown 上のインデックス（選択順）
    selected_novelties = []   # 各ステップで選択時のnoveltyスコア
    selected_pred_mean = []   # 予測値
    selected_pred_var  = []   # 予測分散

    print(f"Sequential Greedy バッチ選択 (N_BATCH={N_BATCH})")
    print("-" * 50)

    for step in range(N_BATCH):
        X_remaining = X_unknown[remaining_mask]
        remaining_original_idx = np.where(remaining_mask)[0]  # X_unknown 上の元インデックス

        novelty, pred_mean, pred_var, density, bw = stein_novelty(
            X_remaining, X_ref_virtual, final_model, bandwidth_mode=STEIN_KERNEL_BANDWIDTH
        )

        best_local_idx  = np.argmax(novelty)             # remaining 内でのインデックス
        best_global_idx = remaining_original_idx[best_local_idx]  # X_unknown 上のインデックス

        selected_indices.append(best_global_idx)
        selected_novelties.append(novelty[best_local_idx])
        selected_pred_mean.append(pred_mean[best_local_idx])
        selected_pred_var.append(pred_var[best_local_idx])

        # 選んだ点を仮の既知点として追加し、候補から除外
        X_ref_virtual = np.vstack([X_ref_virtual, X_unknown[best_global_idx]])
        remaining_mask[best_global_idx] = False

        print(f"Step {step+1:2d}: 候補インデックス={best_global_idx:4d}  "
              f"Novelty={novelty[best_local_idx]:.4g}  "
              f"Predicted={pred_mean[best_local_idx]:.4f}  "
              f"Bandwidth={bw:.4f}")

    print("-" * 50)
    print("Sequential Greedy 選択完了")

else:
    print("未測定データがないため、バッチ選択をスキップします。")
    selected_indices = []

## 7. 結果のまとめと保存

2種類の結果を出力します。

- **`batch_selected.csv`**: Sequential Greedy で選んだ N_BATCH 件（選択順）
- **`all_candidates_novelty.csv`**: 未測定全候補の novelty スコア一覧（Step 1 時点の単純ランキング）

In [ ]:
if len(selected_indices) > 0:
    # --- バッチ選択結果 ---
    batch_df = df_unknown.iloc[selected_indices][FEATURE_COLS].copy().reset_index(drop=True)
    batch_df.insert(0, 'Selection_Order', np.arange(1, len(selected_indices) + 1))
    batch_df['Predicted_' + TARGET_COL] = selected_pred_mean
    batch_df['Predictive_Variance']     = selected_pred_var
    batch_df['Stein_Novelty_at_Selection'] = selected_novelties

    batch_csv = os.path.join(OUTPUT_DIR, "batch_selected.csv")
    batch_df.to_csv(batch_csv, index=False)
    print(f"バッチ選択結果を保存しました: {batch_csv}")
    print()
    display(batch_df)

    # --- 全候補の novelty スコア（Step 1 時点、単純ランキング参考用）---
    nov_all, pm_all, pv_all, den_all, _ = stein_novelty(
        X_unknown, X_known, final_model, bandwidth_mode=STEIN_KERNEL_BANDWIDTH
    )
    all_df = df_unknown[FEATURE_COLS].copy()
    all_df['Predicted_' + TARGET_COL] = pm_all
    all_df['Predictive_Variance']     = pv_all
    all_df['Kernel_Density_to_Known'] = den_all
    all_df['Stein_Novelty']           = nov_all
    all_df = all_df.sort_values('Stein_Novelty', ascending=False).reset_index(drop=True)
    all_df.insert(0, 'Rank', np.arange(1, len(all_df) + 1))

    all_csv = os.path.join(OUTPUT_DIR, "all_candidates_novelty.csv")
    all_df.to_csv(all_csv, index=False)
    print(f"全候補 novelty 一覧を保存しました: {all_csv}")

## 8. 可視化

In [ ]:
if len(selected_indices) > 0:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # --- 左: 全候補の Predicted vs Novelty。選択点を強調表示 ---
    sc = axes[0].scatter(
        all_df['Predicted_' + TARGET_COL], all_df['Stein_Novelty'],
        c=all_df['Predictive_Variance'], cmap='viridis',
        alpha=0.6, edgecolor='k', linewidth=0.3, label='All candidates'
    )
    plt.colorbar(sc, ax=axes[0], label='Predictive Variance')

    # 選択点を重ねてプロット
    axes[0].scatter(
        batch_df['Predicted_' + TARGET_COL], batch_df['Stein_Novelty_at_Selection'],
        color='crimson', s=100, zorder=5, edgecolor='white', linewidth=1.0,
        label=f'Selected (N={N_BATCH})'
    )
    for _, row in batch_df.iterrows():
        axes[0].annotate(
            f"#{int(row['Selection_Order'])}",
            (row['Predicted_' + TARGET_COL], row['Stein_Novelty_at_Selection']),
            textcoords='offset points', xytext=(5, 5), fontsize=8, color='crimson'
        )
    axes[0].set_xlabel(f'Predicted {TARGET_COL}')
    axes[0].set_ylabel('Stein Novelty')
    axes[0].set_title('Predicted Value vs Novelty\n(red: Sequential Greedy selected)')
    axes[0].legend()
    axes[0].grid(True, linestyle='--', alpha=0.5)

    # --- 右: 選択された N_BATCH 件の選択順 vs Novelty ---
    axes[1].bar(
        batch_df['Selection_Order'], batch_df['Stein_Novelty_at_Selection'],
        color='crimson', alpha=0.8, edgecolor='k', linewidth=0.5
    )
    axes[1].set_xlabel('Selection Order')
    axes[1].set_ylabel('Stein Novelty (at selection step)')
    axes[1].set_title(f'Sequential Greedy: Novelty per Step (N_BATCH={N_BATCH})')
    axes[1].set_xticks(batch_df['Selection_Order'])
    axes[1].grid(True, axis='y', linestyle='--', alpha=0.5)

    plt.tight_layout()
    plot_file = os.path.join(OUTPUT_DIR, "batch_selection_plot.png")
    plt.savefig(plot_file, dpi=150)
    plt.show()
    plt.close()
    print(f"保存しました: {plot_file}")

else:
    print("未測定データがないため、プロットはスキップします。")

## 9. 次の実験候補（選択結果）

`Selection_Order` が小さいほど、**そのステップ時点で最も高い Stein Novelty を持つ点**です。
Sequential Greedy により各候補は互いに空間的に離れることが保証されています。

In [ ]:
if len(selected_indices) > 0:
    display(batch_df)